# 01. AI Hub 원본 데이터 확인 및 재현 가능한 raw 샘플링


## 목적

AI Hub의 공식 **Training / Validation 구분을 그대로 유지**하면서 금속캔, 페트병, 플라스틱의 오염 상태별 샘플을 추출합니다.

- 데이터셋: [재활용품 분류 및 선별 데이터](https://www.aihub.or.kr/aihubdata/data/view.do?dataSetSn=71362)
- 난수 시드: `42`
- 대상 오염 상태: `오염없음`, `이물질(외부)`, `이물질(내부)`
- `이물질(전체)` 및 오타 값은 제외
- 실행 결과와 제외 사유를 JSON/CSV로 기록

> 원본 프로젝트에서 사용한 코드를 정리한 재현용 버전입니다. 기존 데이터 보호를 위해 기본 출력 위치는 `reproduced_dataset/raw`입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter, defaultdict
import csv
import json
import random
import shutil
import zipfile

from tqdm.auto import tqdm


## 1. 경로와 샘플 수 설정

본인의 Drive 구조가 다르면 아래 두 경로만 수정합니다. `OUTPUT_ROOT`는 기존 `test_dataset/raw`가 아니라 별도 재현 폴더를 사용합니다.


In [ ]:
TEAM_PROJECT = Path('/content/drive/MyDrive/TeamProject')
AIHUB_ROOT = TEAM_PROJECT / '재활용품 분류 및 선별 데이터' / '정식개방데이터'
OUTPUT_ROOT = TEAM_PROJECT / 'test_dataset' / 'reproduced_dataset'
RAW_ROOT = OUTPUT_ROOT / 'raw'
META_ROOT = OUTPUT_ROOT / 'meta'

SOURCE_DIRS = {
    'train': {
        'images': AIHUB_ROOT / 'Training' / '01.원천데이터',
        'labels': AIHUB_ROOT / 'Training' / '02.라벨링데이터',
    },
    'val': {
        'images': AIHUB_ROOT / 'Validation' / '01.원천데이터',
        'labels': AIHUB_ROOT / 'Validation' / '02.라벨링데이터',
    },
}

RANDOM_SEED = 42

# 최종 프로젝트에서 사용한 목표 수량. 후보가 부족하면 가능한 수량까지만 사용합니다.
TARGET_COUNTS = {
    'train': {
        ('금속캔', '오염없음'): 2500,
        ('금속캔', '이물질(외부)'): 1500,
        ('금속캔', '이물질(내부)'): 544,
        ('페트병', '오염없음'): 2500,
        ('페트병', '이물질(외부)'): 1500,
        ('페트병', '이물질(내부)'): 767,
        ('플라스틱', '오염없음'): 2500,
        ('플라스틱', '이물질(외부)'): 1500,
        ('플라스틱', '이물질(내부)'): 1000,
    },
    'val': {
        ('금속캔', '오염없음'): 500,
        ('금속캔', '이물질(외부)'): 200,
        ('금속캔', '이물질(내부)'): 69,
        ('페트병', '오염없음'): 500,
        ('페트병', '이물질(외부)'): 200,
        ('페트병', '이물질(내부)'): 91,
        ('플라스틱', '오염없음'): 700,
        ('플라스틱', '이물질(외부)'): 500,
        ('플라스틱', '이물질(내부)'): 399,
    },
}

# True로 바꾸기 전에는 기존 재현 결과를 삭제하지 않습니다.
OVERWRITE = False

for split, dirs in SOURCE_DIRS.items():
    for kind, path in dirs.items():
        assert path.is_dir(), f'{split}/{kind} 경로를 찾지 못했습니다: {path}'

print('출력 위치:', OUTPUT_ROOT)


## 2. AI Hub ZIP 인덱싱

이미지와 JSON의 파일 stem을 기준으로 연결합니다. 동일 stem이 중복되면 자동으로 숨기지 않고 오류로 중단합니다.


In [ ]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def zip_files(directory):
    return sorted(directory.glob('*.zip'))

def build_zip_index(directory, suffixes, desc):
    index = {}
    duplicates = defaultdict(list)
    archives = zip_files(directory)

    for archive_path in tqdm(archives, desc=desc):
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.namelist():
                member_path = Path(member)
                if member_path.suffix.lower() not in suffixes:
                    continue
                stem = member_path.stem
                location = (archive_path, member)
                if stem in index:
                    duplicates[stem].append(location)
                else:
                    index[stem] = location

    if duplicates:
        raise RuntimeError(f'{desc}: 중복 stem {len(duplicates)}개 발견')
    return index

image_indexes = {}
label_indexes = {}

for split, dirs in SOURCE_DIRS.items():
    image_indexes[split] = build_zip_index(
        dirs['images'], IMAGE_SUFFIXES, f'{split} 이미지 인덱싱'
    )
    label_indexes[split] = build_zip_index(
        dirs['labels'], {'.json'}, f'{split} JSON 인덱싱'
    )
    print(split, 'images=', len(image_indexes[split]), 'json=', len(label_indexes[split]))


## 3. 후보 수집 및 계층별 샘플링

각 JSON에서 대상 재질·오염 상태에 해당하는 `BOX` annotation을 찾습니다. 이미지와 JSON이 모두 존재하는 샘플만 후보로 사용합니다.


In [ ]:
TARGET_MATERIALS = {'금속캔', '페트병', '플라스틱'}
TARGET_DIRTINESS = {'오염없음', '이물질(외부)', '이물질(내부)'}

def normalize_dirtiness(value):
    value = '' if value is None else str(value).strip()
    aliases = {
        '외부오염': '이물질(외부)',
        '내부오염': '이물질(내부)',
        '내용물(내부)': '이물질(내부)',
        '오엽없음': None,
        '이물질(전체)': None,
    }
    return aliases.get(value, value)

def load_json(location):
    archive_path, member = location
    with zipfile.ZipFile(archive_path) as archive:
        with archive.open(member) as stream:
            return json.load(stream)

def collect_candidates(split):
    groups = defaultdict(list)
    skipped = Counter()

    for stem, json_location in tqdm(label_indexes[split].items(), desc=f'{split} 후보 수집'):
        if stem not in image_indexes[split]:
            skipped['image_not_found'] += 1
            continue
        try:
            data = load_json(json_location)
        except Exception:
            skipped['json_read_error'] += 1
            continue

        keys = []
        for ann in data.get('ANNOTATION_INFO', []):
            material = str(ann.get('CLASS', '')).strip()
            dirtiness = normalize_dirtiness(ann.get('DIRTINESS'))
            shape_type = str(ann.get('SHAPE_TYPE', '')).strip().upper()
            if material in TARGET_MATERIALS and dirtiness in TARGET_DIRTINESS and shape_type == 'BOX':
                keys.append((material, dirtiness))

        unique_keys = sorted(set(keys))
        if not unique_keys:
            skipped['no_target_box'] += 1
            continue
        if len(unique_keys) > 1:
            skipped['mixed_target_classes'] += 1
            continue

        groups[unique_keys[0]].append(stem)

    return groups, skipped

rng = random.Random(RANDOM_SEED)
selected = {}
audit = {'random_seed': RANDOM_SEED, 'splits': {}}

for split in ('train', 'val'):
    groups, skipped = collect_candidates(split)
    selected[split] = []
    audit['splits'][split] = {'skipped': dict(skipped), 'groups': {}}

    for key, target in TARGET_COUNTS[split].items():
        stems = sorted(groups.get(key, []))
        rng.shuffle(stems)
        chosen = stems[:min(target, len(stems))]
        selected[split].extend((stem, key) for stem in chosen)
        audit['splits'][split]['groups']['|'.join(key)] = {
            'available': len(stems), 'target': target, 'selected': len(chosen)
        }

    selected[split].sort(key=lambda item: item[0])
    print(split, '선택:', len(selected[split]))


## 4. raw 데이터 저장

기존 출력이 있으면 기본적으로 중단합니다. 삭제 후 재생성하려는 경우에만 설정 셀의 `OVERWRITE=True`로 변경합니다.


In [ ]:
if RAW_ROOT.exists():
    has_files = any(path.is_file() for path in RAW_ROOT.rglob('*'))
    if has_files and not OVERWRITE:
        raise FileExistsError(
            f'기존 결과가 있습니다: {RAW_ROOT}\n'
            '덮어쓰려면 설정 셀에서 OVERWRITE=True로 변경하세요.'
        )
    if OVERWRITE:
        shutil.rmtree(RAW_ROOT)

for split in ('train', 'val'):
    (RAW_ROOT / split / 'images').mkdir(parents=True, exist_ok=True)
    (RAW_ROOT / split / 'labels_json').mkdir(parents=True, exist_ok=True)
META_ROOT.mkdir(parents=True, exist_ok=True)

def extract_member(location, destination):
    archive_path, member = location
    with zipfile.ZipFile(archive_path) as archive:
        with archive.open(member) as source, open(destination, 'wb') as target:
            shutil.copyfileobj(source, target)

manifest_rows = []
for split in ('train', 'val'):
    for stem, (material, dirtiness) in tqdm(selected[split], desc=f'{split} 저장'):
        image_archive, image_member = image_indexes[split][stem]
        json_archive, json_member = label_indexes[split][stem]
        image_name = Path(image_member).name
        json_name = Path(json_member).name

        extract_member((image_archive, image_member), RAW_ROOT / split / 'images' / image_name)
        extract_member((json_archive, json_member), RAW_ROOT / split / 'labels_json' / json_name)
        manifest_rows.append({
            'split': split, 'stem': stem, 'material': material, 'dirtiness': dirtiness,
            'image_name': image_name, 'json_name': json_name,
        })

with open(META_ROOT / 'raw_manifest.csv', 'w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(manifest_rows[0]))
    writer.writeheader()
    writer.writerows(manifest_rows)

with open(META_ROOT / 'raw_sampling_meta.json', 'w', encoding='utf-8') as stream:
    json.dump(audit, stream, ensure_ascii=False, indent=2)

print('raw 생성 완료:', RAW_ROOT)


## 5. 결과 무결성 검사

이미지와 JSON의 stem 집합, Train/Validation 중복, 클래스별 선택 수량을 확인합니다.


In [ ]:
summary = {}
split_stems = {}

for split in ('train', 'val'):
    image_stems = {p.stem for p in (RAW_ROOT / split / 'images').iterdir() if p.is_file()}
    json_stems = {p.stem for p in (RAW_ROOT / split / 'labels_json').glob('*.json')}
    split_stems[split] = image_stems
    class_counts = Counter((material, dirtiness) for _, (material, dirtiness) in selected[split])

    summary[split] = {
        'images': len(image_stems),
        'jsons': len(json_stems),
        'missing_json': sorted(image_stems - json_stems),
        'missing_image': sorted(json_stems - image_stems),
        'class_counts': {'|'.join(key): value for key, value in sorted(class_counts.items())},
    }

summary['train_val_stem_overlap'] = len(split_stems['train'] & split_stems['val'])

assert not summary['train']['missing_json'] and not summary['train']['missing_image']
assert not summary['val']['missing_json'] and not summary['val']['missing_image']
assert summary['train_val_stem_overlap'] == 0

with open(META_ROOT / 'raw_validation_summary.json', 'w', encoding='utf-8') as stream:
    json.dump(summary, stream, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
